<a href="https://colab.research.google.com/github/aishwaryarajendracs/AI_LAB/blob/main/1WA24CS027_Week8_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Forward Chaining in First Order Logic

# Initial facts
facts = [
    ("American", "Robert"),
    ("Missile", "T1"),
    ("Owns", "A", "T1"),
    ("Enemy", "A", "America")
]


# Rules: (conditions, conclusion)
rules = [

    # Missiles are weapons
    (
        [("Missile", "x")],
        ("Weapon", "x")
    ),

    # All missiles owned by A were sold by Robert to A
    (
        [("Missile", "x"), ("Owns", "A", "x")],
        ("Sells", "Robert", "x", "A")
    ),

    # Enemy of America is hostile
    (
        [("Enemy", "x", "America")],
        ("Hostile", "x")
    ),

    # Selling weapons to hostile nations is a crime
    (
        [
            ("American", "p"),
            ("Weapon", "q"),
            ("Sells", "p", "q", "r"),
            ("Hostile", "r")
        ],
        ("Criminal", "p")
    )
]


# Match a rule pattern with a known fact
def match(pattern, fact, substitution):

    if pattern[0] != fact[0]:
        return None

    if len(pattern) != len(fact):
        return None

    subst = substitution.copy()

    for p, f in zip(pattern[1:], fact[1:]):

        if isinstance(p, str) and p[0].islower():

            if p in subst:
                if subst[p] != f:
                    return None

            else:
                subst[p] = f

        elif p != f:
            return None

    return subst


# Find all possible matches for rule conditions
def find_matches(conditions, facts, substitution=None):

    if substitution is None:
        substitution = {}

    if not conditions:
        return [substitution]

    results = []

    first = conditions[0]
    remaining = conditions[1:]

    for fact in facts:

        new_subst = match(first, fact, substitution)

        if new_subst is not None:

            results.extend(
                find_matches(remaining, facts, new_subst)
            )

    return results


# Apply substitution to a conclusion
def apply_substitution(expression, substitution):

    return tuple(
        substitution.get(item, item)
        if isinstance(item, str) else item
        for item in expression
    )


# Forward Chaining Algorithm
def forward_chaining(query):

    iteration = 1

    while True:

        new_facts = []

        print("\nIteration:", iteration)

        for conditions, conclusion in rules:

            substitutions = find_matches(conditions, facts)

            for subst in substitutions:

                new_fact = apply_substitution(
                    conclusion, subst
                )

                if new_fact not in facts and new_fact not in new_facts:

                    new_facts.append(new_fact)

                    print("New Fact Derived:", new_fact)

        if not new_facts:
            break

        facts.extend(new_facts)

        if query in facts:
            print("\nQuery Proved:", query)
            return True

        iteration += 1

    if query in facts:
        print("\nQuery Proved:", query)
        return True

    print("\nQuery Cannot Be Proved:", query)
    return False


# Query
query = ("Criminal", "Robert")

print("Initial Facts:")
for fact in facts:
    print(fact)

forward_chaining(query)

print("\nFinal Knowledge Base:")
for fact in facts:
    print(fact)


Initial Facts:
('American', 'Robert')
('Missile', 'T1')
('Owns', 'A', 'T1')
('Enemy', 'A', 'America')

Iteration: 1
New Fact Derived: ('Weapon', 'T1')
New Fact Derived: ('Sells', 'Robert', 'T1', 'A')
New Fact Derived: ('Hostile', 'A')

Iteration: 2
New Fact Derived: ('Criminal', 'Robert')

Query Proved: ('Criminal', 'Robert')

Final Knowledge Base:
('American', 'Robert')
('Missile', 'T1')
('Owns', 'A', 'T1')
('Enemy', 'A', 'America')
('Weapon', 'T1')
('Sells', 'Robert', 'T1', 'A')
('Hostile', 'A')
('Criminal', 'Robert')
